In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Rutas del proyecto: el notebook corre desde notebooks/, así que las rutas
# relativas no sirven. `rutas` las resuelve desde la raíz del repo.
from geostats import rutas

In [ ]:
atus = pd.read_csv(rutas.RATIV, low_memory=False)

In [7]:
atus.sample(29)

        MUNICIPIO  ENCUESTA  Día  Mes   Año  Día de la semana  \
19646          46   2464705   14    2  2025                 6   
243825         18      1685   23    6  2022                 5   
96158          18   5181317   11    5  2024                 7   
143895         39   1393283   16    1  2023                 2   
34195          39   5391606    8    5  2025                 5   
139851         46  12464630    1   12  2024                 1   
79798          39   2391963    4    2  2024                 1   
259643         46      5330   24    8  2022                 4   
229740         48      6169   17    3  2022                 5   
136453         39  12391232    4   12  2024                 4   
148071         18   2181432   12    2  2023                 1   
44816          39   7391735   14    7  2025                 2   
262282         39      2053    3    9  2022                 7   
183384         19   8190578   24    8  2023                 5   
235656         46      54

In [ ]:
# Los CSV georreferenciados vienen en latin-1, NO en lo que reporta chardet
# (devuelve CP874/TIS-620 con confianza 0.00 porque casi todo es ASCII).
# `geostats.consolidar` ya los une, tipa y valida; genera el parquet una vez:
#     uv run python -m geostats.consolidar

atus_geo = pd.read_parquet(rutas.ATUS_GEORREFERENCIADO)
atus_geo.shape

In [ ]:
# `ID` NO es único: en 2019-2020 es un folio por municipio. La llave es esta.
assert not atus_geo.duplicated(["ANIO", "EDO", "MPIO", "ID"]).any()

# Ojo al comparar años: la cobertura crece de 91 a 198 municipios (2019->2024),
# así que los conteos crudos por año no son comparables sin normalizar.
atus_geo.groupby("ANIO").agg(
    accidentes=("ID", "size"),
    municipios=("CVE_MUN", "nunique"),
    muertos=("TOTMUERTOS", "sum"),
)

In [ ]:
# CVE_MUN (2 dígitos de estado + 3 de municipio) es la llave para unir con el
# marco geoestadístico del INEGI y con los shapefiles de cada año.
puntos = gpd.GeoDataFrame(
    atus_geo,
    geometry=gpd.points_from_xy(atus_geo.LONGITUD, atus_geo.LATITUD),
    crs="EPSG:4326",
)
puntos.head()